In [0]:
from pyspark.sql.functions import col,trim,lower,initcap,when,current_timestamp,row_number
from pyspark.sql.window import Window

bronze_df = spark.read.table("`reconciliation-platform`.bronze.customers")

cleaned_df = (bronze_df
    .withColumn("customer_id",trim(col("customer_id")))
    .withColumn("customer_name",initcap(trim(col("customer_name"))))
    .withColumn("email",lower(trim(col("email"))))
    .withColumn("city",initcap(trim(col("city"))))
    .withColumn("state",initcap(trim(col("state"))))
    .withColumn("customer_segment",lower(trim(col("customer_segment"))))
    .withColumn("acquisition_channel",lower(trim(col("acquisition_channel"))))
    .withColumn("signup_date",col("signup_date").cast("date"))
    .withColumn("updated_at",col("updated_at").cast("timestamp"))
    .withColumn("customer_segment",when(col("customer_segment").isNull() | (col("customer_segment") == ""),"unknown").otherwise(col("customer_segment")))
    .withColumn("acquisition_channel",when(col("acquisition_channel").isNull() | (col("acquisition_channel") == ""),"unknown").otherwise(col("acquisition_channel")))
    .filter(col("customer_id").isNotNull())
    .filter(col("customer_id") != "")
)

window_spec = Window.partitionBy("customer_id").orderBy(col("updated_at").desc_nulls_last(),col("_ingestion_timestamp").desc())

silver_df = (cleaned_df
    .withColumn("_row_number",row_number().over(window_spec))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
    .withColumn("_silver_processed_at",current_timestamp())
)

(silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("`reconciliation-platform`.silver.customers"))